# LSA TESTING

In [3]:
from pathlib import Path
from preprocessing import clean_pan_source_text, chunk_clean_text


path = Path("../datasets/PAN2011/usable/source-document/part1/source-document00001.txt")

raw_text = path.read_text(encoding="utf-8", errors="ignore")
clean_text = clean_pan_source_text(raw_text)

print("RAW LENGTH:", len(raw_text))
print("CLEAN LENGTH:", len(clean_text))
print("REMOVED:", len(raw_text) - len(clean_text))

print("\n--- RAW SAMPLE ---")
print(raw_text[:1500])

print("\n--- CLEAN SAMPLE ---")
print(clean_text[:1500])

RAW LENGTH: 251039
CLEAN LENGTH: 250031
REMOVED: 1008

--- RAW SAMPLE ---
﻿On the 18th, we finished the journey by a nine mile march to Bocking, and there settled down
into billets for the rest of our time in England. Though we were spoilt at Harpenden, we are
sure that all ranks have nothing but pleasant recollections of the time spent at Braintree
and Bocking, where one and all treated us with the greatest kindness, and we hope were sorry
to lose us. Where all were so kind it is almost invidious to mention names, but one feels (though
they themselves would be the first to deny it) that a special debt of gratitude is owed to
the Nuns of the Convent at Booking, whose kindness and care for those who were billeted at
the Convent, and for all with whom they came in contact, were beyond all praise.

In order to prepare for any possible German landing on the Essex coast orders had been issued
for a series of trenches to be dug to form defensive lines for the protection of London, and
we wer

In [7]:
import re
from typing import List, Dict


def chunk_clean_text( clean_text: str, chunk_size: int = 180, overlap: int = 50, min_words: int = 40) -> List[Dict]:
    """
    Split cleaned text into overlapping word chunks.

    This is used before:
    - LSA
    - ESA
    - embedding/vector database indexing
    """

    tokens = list(re.finditer(r"\S+", clean_text))

    if not tokens:
        return []

    chunks = []
    step = max(1, chunk_size - overlap)

    for start_word in range(0, len(tokens), step):
        end_word = min(start_word + chunk_size, len(tokens))

        if end_word - start_word < min_words:
            continue

        start_char = tokens[start_word].start()
        end_char = tokens[end_word - 1].end()

        chunks.append({
            "chunk_text": clean_text[start_char:end_char],
            "start_char": start_char,
            "end_char": end_char,
            "word_start": start_word,
            "word_end": end_word,
        })

    return chunks

Different chunk sizes were evaluated because passage length influences the balance between contextual semantic representation and localization accuracy. Smaller chunks improve fine-grained alignment but may lose semantic context, while larger chunks provide more context but reduce the precision of detected passage boundaries.

In [ ]:
from typing import Dict
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer


from preprocessing import clean_pan_source_text, chunk_clean_text


class LSABranch:
    """
    Branch A:
    Word chunks -> TF-IDF -> TruncatedSVD -> LSA vectors -> cosine similarity
    """

    def __init__(
        self,
        chunk_size: int = 180,
        overlap: int = 50,
        min_words: int = 40,
        n_components: int = 200,
        top_k: int = 5,
    ):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.min_words = min_words
        self.n_components = n_components
        self.top_k = top_k

        self.vectorizer = None
        self.lsa_model = None
        self.source_vectors = None
        self.source_metadata = None

    def build_index(self, source_docs: Dict[str, str]) -> pd.DataFrame:
        """
        Build LSA index from source documents.

        source_docs format:
        {
            "source-document00001.txt": "raw text...",
            "source-document00002.txt": "raw text..."
        }
        """

        rows = []
        source_chunks_text = []

        for source_doc_id, raw_text in source_docs.items():
            clean_text = light_clean_source_text(raw_text)

            chunks = chunk_clean_text(
                clean_text,
                chunk_size=self.chunk_size,
                overlap=self.overlap,
                min_words=self.min_words,
            )

            for chunk_id, chunk in enumerate(chunks):
                source_chunks_text.append(chunk["chunk_text"])

                rows.append({
                    "source_doc": source_doc_id,
                    "source_chunk_id": chunk_id,
                    "source_start_char": chunk["start_char"],
                    "source_end_char": chunk["end_char"],
                    "source_text": chunk["chunk_text"],
                })

        if not source_chunks_text:
            raise ValueError("No source chunks created for LSA.")

        self.source_metadata = pd.DataFrame(rows)

        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True,
            max_features=100_000,
        )

        tfidf_matrix = self.vectorizer.fit_transform(source_chunks_text)

        safe_components = min(
            self.n_components,
            tfidf_matrix.shape[0] - 1,
            tfidf_matrix.shape[1] - 1,
        )

        safe_components = max(2, safe_components)

        self.lsa_model = make_pipeline(
            TruncatedSVD(n_components=safe_components, random_state=42),
            Normalizer(copy=False),
        )

        self.source_vectors = self.lsa_model.fit_transform(tfidf_matrix)

        return self.source_metadata

    def retrieve(
        self,
        suspicious_doc_id: str,
        suspicious_text: str,
        threshold: float = 0.30,
    ) -> pd.DataFrame:
        """
        Retrieve candidate source chunks for one suspicious document.
        """

        if self.vectorizer is None or self.lsa_model is None:
            raise RuntimeError("LSA index has not been built yet.")

        clean_text = light_clean_source_text(suspicious_text)

        suspicious_chunks = chunk_clean_text(
            clean_text,
            chunk_size=self.chunk_size,
            overlap=self.overlap,
            min_words=self.min_words,
        )

        if not suspicious_chunks:
            return pd.DataFrame()

        suspicious_chunk_texts = [chunk["chunk_text"] for chunk in suspicious_chunks]

        suspicious_tfidf = self.vectorizer.transform(suspicious_chunk_texts)
        suspicious_vectors = self.lsa_model.transform(suspicious_tfidf)

        similarity_matrix = cosine_similarity(suspicious_vectors, self.source_vectors)

        results = []

        for suspicious_chunk_id, similarities in enumerate(similarity_matrix):
            best_source_indices = np.argsort(similarities)[::-1][:self.top_k]

            for source_index in best_source_indices:
                score = float(similarities[source_index])

                if score < threshold:
                    continue

                source_row = self.source_metadata.iloc[source_index]
                suspicious_chunk = suspicious_chunks[suspicious_chunk_id]

                results.append({
                    "method": "LSA",

                    "suspicious_doc": suspicious_doc_id,
                    "suspicious_chunk_id": suspicious_chunk_id,
                    "suspicious_start_char": suspicious_chunk["start_char"],
                    "suspicious_end_char": suspicious_chunk["end_char"],
                    "suspicious_text": suspicious_chunk["chunk_text"],

                    "source_doc": source_row["source_doc"],
                    "source_chunk_id": int(source_row["source_chunk_id"]),
                    "source_start_char": int(source_row["source_start_char"]),
                    "source_end_char": int(source_row["source_end_char"]),
                    "source_text": source_row["source_text"],

                    "score": round(score, 4),
                })

        if not results:
            return pd.DataFrame()

        return pd.DataFrame(results).sort_values("score", ascending=False)

In [5]:
from pathlib import Path
import pandas as pd

from branches import LSABranch


def load_txt_folder(folder_path: str):
    folder = Path(folder_path)
    docs = {}

    for path in folder.rglob("*.txt"):
        doc_id = path.name
        docs[doc_id] = path.read_text(encoding="utf-8", errors="ignore")

    return docs


source_docs = load_txt_folder("../data/external-detection-corpus/source-document")
suspicious_docs = load_txt_folder("../data/external-detection-corpus/suspicious-document")

print("Source docs:", len(source_docs))
print("Suspicious docs:", len(suspicious_docs))

suspicious_doc_id = list(suspicious_docs.keys())[0]
suspicious_text = suspicious_docs[suspicious_doc_id]

lsa = LSABranch(
    chunk_size=180,
    overlap=50,
    min_words=40,
    n_components=200,
    top_k=5,
)

source_metadata = lsa.build_index(source_docs)

print("Indexed source chunks:", len(source_metadata))

lsa_results = lsa.retrieve(
    suspicious_doc_id=suspicious_doc_id,
    suspicious_text=suspicious_text,
    threshold=0.30,
)

print("Results:", len(lsa_results))

lsa_results[
    ["method", "score", "suspicious_doc", "suspicious_chunk_id", "source_doc", "source_chunk_id"]
].head(20)

Source docs: 0
Suspicious docs: 0


IndexError: list index out of range

In [4]:
import importlib
import preprocessing
from pathlib import Path
# 
importlib.reload(preprocessing)

DATASET_ROOT = Path("../datasets/PAN2011/usable")

SOURCE_FOLDER = DATASET_ROOT / "source-document"
SUSPICIOUS_FOLDER = DATASET_ROOT / "suspicious-document"

OUTPUT_FOLDER = Path("processed_files")
OUTPUT_FOLDER.mkdir(exist_ok=True)

SOURCE_OUTPUT = OUTPUT_FOLDER / "source_documents.parquet"
SUSPICIOUS_OUTPUT = OUTPUT_FOLDER / "suspicious_documents.parquet"

preprocessing.source_doc_processing(SOURCE_FOLDER,SUSPICIOUS_FOLDER,SOURCE_OUTPUT,SUSPICIOUS_OUTPUT,100)

Found 11093 .txt files in ..\datasets\PAN2011\usable\source-document
[1/11093] Reading: part1/source-document00001.txt
[500/11093] Reading: part1/source-document00500.txt
[1000/11093] Reading: part10/source-document05000.txt
[1500/11093] Reading: part11/source-document05500.txt
[2000/11093] Reading: part12/source-document06000.txt
[2500/11093] Reading: part13/source-document06500.txt
[3000/11093] Reading: part14/source-document07000.txt
[3500/11093] Reading: part15/source-document07500.txt
[4000/11093] Reading: part16/source-document08000.txt
[4500/11093] Reading: part17/source-document08500.txt
[5000/11093] Reading: part18/source-document09000.txt
[5500/11093] Reading: part19/source-document09500.txt
[6000/11093] Reading: part2/source-document01000.txt
[6500/11093] Reading: part20/source-document10000.txt
[7000/11093] Reading: part21/source-document10500.txt
[7500/11093] Reading: part22/source-document11000.txt
[8000/11093] Reading: part3/source-document01407.txt
[8500/11093] Reading:

In [ ]:
import pandas as pd

source_df = pd.read_parquet("processed_files/source_documents.parquet")
suspicious_df = pd.read_parquet("processed_files/suspicious_documents.parquet")

source_df.head()

,doc_id,relative_path,part,text,char_count,word_count
0,source-document00001.txt,part1/source-document00001.txt,part1,"﻿On the 18th, we finished the journey by a nin...",251039,43689
1,source-document00002.txt,part1/source-document00002.txt,part1,﻿THE YOUNG LADY'S EQUESTRIAN MANUAL.\n\nE.LAND...,90734,15965
2,source-document00003.txt,part1/source-document00003.txt,part1,﻿MORE PAGES FROM A JOURNAL WITH OTHER PAPERS\n...,305185,54993
3,source-document00004.txt,part1/source-document00004.txt,part1,﻿D.W.]\n\nCONFESSION OF A CHILD OF THE CENTURY...,175758,32373
4,source-document00005.txt,part1/source-document00005.txt,part1,﻿Now I do not claim for Charles that he is any...,30194,5166


: 

In [ ]:
from pathlib import Path
from typing import Optional

import joblib
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


def load_lsa_esa_chunks(
    path: Path,
    text_column: str = "lsa_esa_text",
) -> pd.DataFrame:
    df = pd.read_parquet(path)

    required_columns = {
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
        text_column,
    }

    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df.copy()
    df[text_column] = df[text_column].fillna("").astype(str)

    # Remove empty chunks after normalization.
    df = df[df[text_column].str.strip() != ""].reset_index(drop=True)

    return df


def build_lsa_index(
    source_chunks_path: Path,
    artifact_dir: Path,
    text_column: str = "lsa_esa_text",
    n_components: int = 200,
    max_features: int = 100_000,
    max_source_chunks: Optional[int] = None,
):
    """
    Build LSA index from source chunks.

    Outputs:
    - tfidf_vectorizer.joblib
    - svd_model.joblib
    - source_lsa_vectors.npy
    - source_lsa_metadata.parquet
    """

    artifact_dir = Path(artifact_dir)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    source_df = load_lsa_esa_chunks(
        path=source_chunks_path,
        text_column=text_column,
    )

    if max_source_chunks is not None:
        source_df = source_df.head(max_source_chunks).reset_index(drop=True)

    source_texts = source_df[text_column].tolist()
    total_source_chunks = len(source_df)

    print(f"Loaded {total_source_chunks} source chunks")
    print("Fitting TF-IDF...")

    vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=2,
        max_df=0.95,
        ngram_range=(1, 1),  # use unigrams first; safer and faster for large corpus
        sublinear_tf=True,
        norm="l2",
    )

    source_tfidf = vectorizer.fit_transform(source_texts)

    print(f"TF-IDF matrix shape: {source_tfidf.shape}")

    feature_count = source_tfidf.shape[1]
    actual_components = min(n_components, feature_count - 1)

    if actual_components < 2:
        raise ValueError(
            f"Not enough TF-IDF features for LSA. Feature count: {feature_count}"
        )

    print(f"Fitting TruncatedSVD with {actual_components} components...")

    svd = TruncatedSVD(
        n_components=actual_components,
        random_state=42,
        n_iter=5,
    )

    source_lsa = svd.fit_transform(source_tfidf)

    print("Normalizing LSA vectors...")
    source_lsa = normalize(source_lsa, norm="l2", axis=1)

    metadata_columns = [
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
    ]

    optional_columns = ["file_name", "relative_path", "part", "word_count"]
    metadata_columns += [col for col in optional_columns if col in source_df.columns]

    source_metadata = source_df[metadata_columns].copy()

    print("Saving LSA artifacts...")

    vectors_path = artifact_dir / "source_lsa_vectors.npy"
    metadata_path = artifact_dir / "source_lsa_metadata.parquet"
    svd_path = artifact_dir / "svd_model.joblib"
    vectorizer_path = artifact_dir / "tfidf_vectorizer.joblib"

    print(f"Saving vectors to {vectors_path}...")
    np.save(vectors_path, source_lsa)
    print("Saved source_lsa_vectors.npy")

    print(f"Saving metadata to {metadata_path}...")
    source_metadata.to_parquet(metadata_path, index=False)
    print("Saved source_lsa_metadata.parquet")

    print(f"Saving SVD model to {svd_path}...")
    joblib.dump(svd, svd_path)
    print("Saved svd_model.joblib")

    print(f"Saving TF-IDF vectorizer to {vectorizer_path}...")
    joblib.dump(vectorizer, vectorizer_path)
    print("Saved tfidf_vectorizer.joblib")

    print(f"Saved LSA artifacts to: {artifact_dir}")
    print(f"Explained variance ratio sum: {svd.explained_variance_ratio_.sum():.4f}")

    return vectorizer, svd, source_lsa, source_metadata


def retrieve_lsa_candidates(
    suspicious_chunks_path: Path,
    artifact_dir: Path,
    output_path: Path,
    text_column: str = "lsa_esa_text",
    top_k: int = 20,
    batch_size: int = 128,
    max_suspicious_chunks: Optional[int] = None,
) -> pd.DataFrame:
    """
    Retrieve top-k source chunks for each suspicious chunk using LSA cosine similarity.

    Use max_suspicious_chunks=1 for a quick test.
    """

    artifact_dir = Path(artifact_dir)

    vectorizer_path = artifact_dir / "tfidf_vectorizer.joblib"
    svd_path = artifact_dir / "svd_model.joblib"
    vectors_path = artifact_dir / "source_lsa_vectors.npy"
    metadata_path = artifact_dir / "source_lsa_metadata.parquet"

    for path in [vectorizer_path, svd_path, vectors_path, metadata_path]:
        if not path.exists():
            raise FileNotFoundError(f"Missing LSA artifact: {path}")

    print("Loading LSA artifacts...")
    vectorizer = joblib.load(vectorizer_path)
    svd = joblib.load(svd_path)
    source_lsa = np.load(vectors_path)
    source_metadata = pd.read_parquet(metadata_path)

    suspicious_df = load_lsa_esa_chunks(
        path=suspicious_chunks_path,
        text_column=text_column,
    )

    if max_suspicious_chunks is not None:
        suspicious_df = suspicious_df.head(max_suspicious_chunks).reset_index(drop=True)

    print(f"Loaded {len(suspicious_df)} suspicious chunks")
    print(f"Searching top-{top_k} source chunks per suspicious chunk")

    results = []

    for start in range(0, len(suspicious_df), batch_size):
        end = min(start + batch_size, len(suspicious_df))
        batch_df = suspicious_df.iloc[start:end]

        batch_texts = batch_df[text_column].tolist()

        suspicious_tfidf = vectorizer.transform(batch_texts)
        suspicious_lsa = svd.transform(suspicious_tfidf)
        suspicious_lsa = normalize(suspicious_lsa, norm="l2", axis=1)

        # Cosine similarity because both matrices are normalized.
        similarities = suspicious_lsa @ source_lsa.T

        for local_i, suspicious_row in enumerate(batch_df.itertuples(index=False)):
            sims = similarities[local_i]

            safe_top_k = min(top_k, len(sims))
            top_indices = np.argpartition(-sims, safe_top_k - 1)[:safe_top_k]
            top_indices = top_indices[np.argsort(-sims[top_indices])]

            for rank, source_idx in enumerate(top_indices, start=1):
                source_row = source_metadata.iloc[source_idx]

                results.append({
                    "suspicious_chunk_id": suspicious_row.chunk_id,
                    "suspicious_doc_id": suspicious_row.doc_id,
                    "suspicious_chunk_index": suspicious_row.chunk_index,
                    "suspicious_start_char": suspicious_row.start_char,
                    "suspicious_end_char": suspicious_row.end_char,

                    "source_chunk_id": source_row["chunk_id"],
                    "source_doc_id": source_row["doc_id"],
                    "source_chunk_index": source_row["chunk_index"],
                    "source_start_char": source_row["start_char"],
                    "source_end_char": source_row["end_char"],

                    "LSA_score": float(sims[source_idx]),
                    "LSA_rank": rank,
                })

        print(f"Processed suspicious chunks {start} to {end}")

    output_df = pd.DataFrame(results)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_parquet(output_path, index=False)

    print(f"Saved LSA candidates to: {output_path}")

    return output_df


def test_lsa_retrieval():
    processed_dir = Path("../datasets/processed/PAN2011_300")
    artifact_dir = Path("../artifacts/lsa")

    source_chunks_path = processed_dir / "source_chunks_lsa_esa.parquet"
    suspicious_chunks_path = processed_dir / "suspicious_chunks_lsa_esa.parquet"

    output_path = processed_dir / "lsa_candidates_test.parquet"

    build_lsa_index(
        source_chunks_path=source_chunks_path,
        artifact_dir=artifact_dir,
        n_components=20,
        max_features=20_000,
        max_source_chunks=100000,  # set to 100_000 for faster debug
    )

    candidates = retrieve_lsa_candidates(
        suspicious_chunks_path=suspicious_chunks_path,
        artifact_dir=artifact_dir,
        output_path=output_path,
        top_k=20,
        batch_size=1,
        max_suspicious_chunks=1,
    )

    print(candidates.head(20).to_string())


if __name__ == "__main__":
    test_lsa_retrieval()

Loaded 300000 source chunks
Fitting TF-IDF...
TF-IDF matrix shape: (300000, 20000)
Fitting TruncatedSVD with 20 components...
Normalizing LSA vectors...
Saving LSA artifacts...
Saving vectors to ..\artifacts\lsa\source_lsa_vectors.npy...
Saved source_lsa_vectors.npy
Saving metadata to ..\artifacts\lsa\source_lsa_metadata.parquet...
Saved source_lsa_metadata.parquet
Saving SVD model to ..\artifacts\lsa\svd_model.joblib...
Saved svd_model.joblib
Saving TF-IDF vectorizer to ..\artifacts\lsa\tfidf_vectorizer.joblib...
Saved tfidf_vectorizer.joblib
Saved LSA artifacts to: ..\artifacts\lsa
Explained variance ratio sum: 0.0555
Loading LSA artifacts...
Loaded 1 suspicious chunks
Searching top-20 source chunks per suspicious chunk
Processed suspicious chunks 0 to 1
Saved LSA candidates to: ..\datasets\processed\PAN2011_300\lsa_candidates_test.parquet
                      suspicious_chunk_id                    suspicious_doc_id  suspicious_chunk_index  suspicious_start_char  suspicious_end_char

In [19]:
from pathlib import Path

import pandas as pd

def get_top_source_documents_by_mean_score(
    candidates_path: Path,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    output_path: Path | None = None,
    top_n: int = 20,
    min_match_count: int = 4,
) -> pd.DataFrame:
    """
    For one suspicious document, return the top-N unique source documents
    ranked by the MEAN LSA_score across all candidate chunk matches.

    Only keeps source documents with match_count >= min_match_count.
    """

    candidates_path = Path(candidates_path)
    source_chunks_path = Path(source_chunks_path)

    candidates_df = pd.read_parquet(candidates_path)

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "LSA_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates file: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            mean_LSA_score=("LSA_score", "mean"),
            max_LSA_score=("LSA_score", "max"),
            min_LSA_score=("LSA_score", "min"),
            match_count=("LSA_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = grouped_df.sort_values(
        ["mean_LSA_score", "match_count"],
        ascending=[False, False],
    ).head(top_n).reset_index(drop=True)

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "mean_LSA_score",
            "max_LSA_score",
            "min_LSA_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents by mean score to: {output_path}")

    return grouped_df


if __name__ == "__main__":
    processed_dir = Path("../datasets/processed/PAN2011_300")

    candidates_path = processed_dir / "lsa_candidates_test.parquet"
    source_chunks_path = processed_dir / "source_chunks.parquet"
    output_path = processed_dir / "lsa_top_source_documents_by_mean_score.parquet"

    suspicious_doc_id = "part1__suspicious-document00001.txt"

    top_sources_df = get_top_source_documents_by_mean_score(
        candidates_path=candidates_path,
        source_chunks_path=source_chunks_path,
        suspicious_doc_id=suspicious_doc_id,
        output_path=output_path,
        top_n=20,
        min_match_count=3,
    )

top_sources_df

Saved top source documents by mean score to: ..\datasets\processed\PAN2011_300\lsa_top_source_documents_by_mean_score.parquet


,source_doc_rank,source_doc_id,mean_LSA_score,max_LSA_score,min_LSA_score,match_count,unique_source_chunks,unique_suspicious_chunks,source_relative_path
0,1,part11__source-document05478.txt,0.974561,0.979559,0.966366,5,5,1,part11/source-document05478.txt


In [ ]:
from pathlib import Path
from typing import Optional

import joblib
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


# ============================================================
# CONFIG
# ============================================================

PROCESSED_DIR = Path("../datasets/processed/PAN2011_300")
ARTIFACT_DIR = Path("../artifacts/lsa")

SOURCE_CHUNKS_PATH = PROCESSED_DIR / "source_chunks_lsa_esa.parquet"
SUSPICIOUS_CHUNKS_PATH = PROCESSED_DIR / "suspicious_chunks_lsa_esa.parquet"

SUSPICIOUS_DOC_ID = "part1__suspicious-document00001.txt"

OUTPUT_PATH = PROCESSED_DIR / "lsa_candidates_suspicious_doc_00001.parquet"

# Change this to True only when you want to rebuild the source LSA index.
BUILD_INDEX = False


# ============================================================
# LOAD CHUNKS
# ============================================================

def load_lsa_esa_chunks(
    path: Path,
    text_column: str = "lsa_esa_text",
) -> pd.DataFrame:
    df = pd.read_parquet(path)

    required_columns = {
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
        text_column,
    }

    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df.copy()
    df[text_column] = df[text_column].fillna("").astype(str)
    df = df[df[text_column].str.strip() != ""].reset_index(drop=True)

    return df


# ============================================================
# BUILD LSA INDEX
# ============================================================

def build_lsa_index(
    source_chunks_path: Path,
    artifact_dir: Path,
    text_column: str = "lsa_esa_text",
    n_components: int = 200,
    max_features: int = 100_000,
    max_source_chunks: Optional[int] = None,
):
    """
    Build and save the LSA source index.

    Creates:
    - tfidf_vectorizer.joblib
    - svd_model.joblib
    - source_lsa_vectors.npy
    - source_lsa_metadata.parquet
    """

    artifact_dir = Path(artifact_dir)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    source_df = load_lsa_esa_chunks(
        path=source_chunks_path,
        text_column=text_column,
    )

    if max_source_chunks is not None:
        source_df = source_df.head(max_source_chunks).reset_index(drop=True)

    source_texts = source_df[text_column].tolist()

    print(f"Loaded {len(source_df)} source chunks")
    print("Fitting TF-IDF...")

    vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=2,
        max_df=0.95,
        ngram_range=(1, 1),
        sublinear_tf=True,
        norm="l2",
    )

    source_tfidf = vectorizer.fit_transform(source_texts)

    print(f"TF-IDF matrix shape: {source_tfidf.shape}")

    feature_count = source_tfidf.shape[1]
    actual_components = min(n_components, feature_count - 1)

    if actual_components < 2:
        raise ValueError(
            f"Not enough TF-IDF features for LSA. Feature count: {feature_count}"
        )

    print(f"Fitting TruncatedSVD with {actual_components} components...")

    svd = TruncatedSVD(
        n_components=actual_components,
        random_state=42,
        n_iter=5,
    )

    source_lsa = svd.fit_transform(source_tfidf)

    print("Normalizing source LSA vectors...")
    source_lsa = normalize(source_lsa, norm="l2", axis=1)

    metadata_columns = [
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
    ]

    optional_columns = ["file_name", "relative_path", "part", "word_count"]
    metadata_columns += [col for col in optional_columns if col in source_df.columns]

    source_metadata = source_df[metadata_columns].copy()

    print("Saving LSA artifacts...")

    np.save(artifact_dir / "source_lsa_vectors.npy", source_lsa)
    source_metadata.to_parquet(
        artifact_dir / "source_lsa_metadata.parquet",
        index=False,
    )
    joblib.dump(svd, artifact_dir / "svd_model.joblib")
    joblib.dump(vectorizer, artifact_dir / "tfidf_vectorizer.joblib")

    print(f"Saved LSA artifacts to: {artifact_dir}")
    print(f"Explained variance ratio sum: {svd.explained_variance_ratio_.sum():.4f}")


# ============================================================
# LOAD LSA INDEX
# ============================================================

def load_lsa_index(artifact_dir: Path):
    artifact_dir = Path(artifact_dir)

    vectorizer_path = artifact_dir / "tfidf_vectorizer.joblib"
    svd_path = artifact_dir / "svd_model.joblib"
    vectors_path = artifact_dir / "source_lsa_vectors.npy"
    metadata_path = artifact_dir / "source_lsa_metadata.parquet"

    for path in [vectorizer_path, svd_path, vectors_path, metadata_path]:
        if not path.exists():
            raise FileNotFoundError(f"Missing LSA artifact: {path}")

    print("Loading LSA artifacts...")

    vectorizer = joblib.load(vectorizer_path)
    svd = joblib.load(svd_path)
    source_lsa = np.load(vectors_path)
    source_metadata = pd.read_parquet(metadata_path)

    return vectorizer, svd, source_lsa, source_metadata


# ============================================================
# QUERY ONE SUSPICIOUS DOCUMENT
# ============================================================

def retrieve_lsa_candidates_for_suspicious_doc(
    suspicious_chunks_path: Path,
    artifact_dir: Path,
    suspicious_doc_id: str,
    output_path: Path,
    text_column: str = "lsa_esa_text",
    top_k: int = 20,
    batch_size: int = 8,
    max_suspicious_chunks: Optional[int] = None,
) -> pd.DataFrame:
    """
    Query the LSA source index using only one selected suspicious document.

    Returns top-k source chunks for each suspicious chunk.
    """

    vectorizer, svd, source_lsa, source_metadata = load_lsa_index(artifact_dir)

    suspicious_df = load_lsa_esa_chunks(
        path=suspicious_chunks_path,
        text_column=text_column,
    )

    suspicious_df = suspicious_df[
        suspicious_df["doc_id"] == suspicious_doc_id
    ].copy()

    if suspicious_df.empty:
        raise ValueError(f"No suspicious chunks found for doc_id: {suspicious_doc_id}")

    if max_suspicious_chunks is not None:
        suspicious_df = suspicious_df.head(max_suspicious_chunks).reset_index(drop=True)

    print(f"Selected suspicious document: {suspicious_doc_id}")
    print(f"Suspicious chunks to query: {len(suspicious_df)}")
    print(f"Searching top-{top_k} source chunks per suspicious chunk")

    results = []

    for start in range(0, len(suspicious_df), batch_size):
        end = min(start + batch_size, len(suspicious_df))
        batch_df = suspicious_df.iloc[start:end]

        batch_texts = batch_df[text_column].tolist()

        suspicious_tfidf = vectorizer.transform(batch_texts)
        suspicious_lsa = svd.transform(suspicious_tfidf)
        suspicious_lsa = normalize(suspicious_lsa, norm="l2", axis=1)

        similarities = suspicious_lsa @ source_lsa.T

        for local_i, suspicious_row in enumerate(batch_df.itertuples(index=False)):
            sims = similarities[local_i]

            safe_top_k = min(top_k, len(sims))
            top_indices = np.argpartition(-sims, safe_top_k - 1)[:safe_top_k]
            top_indices = top_indices[np.argsort(-sims[top_indices])]

            for rank, source_idx in enumerate(top_indices, start=1):
                source_row = source_metadata.iloc[source_idx]

                results.append({
                    "suspicious_chunk_id": suspicious_row.chunk_id,
                    "suspicious_doc_id": suspicious_row.doc_id,
                    "suspicious_chunk_index": suspicious_row.chunk_index,
                    "suspicious_start_char": suspicious_row.start_char,
                    "suspicious_end_char": suspicious_row.end_char,

                    "source_chunk_id": source_row["chunk_id"],
                    "source_doc_id": source_row["doc_id"],
                    "source_chunk_index": source_row["chunk_index"],
                    "source_start_char": source_row["start_char"],
                    "source_end_char": source_row["end_char"],

                    "LSA_score": float(sims[source_idx]),
                    "LSA_rank": rank,
                })

        print(f"Processed suspicious chunks {start} to {end}")

    output_df = pd.DataFrame(results)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_parquet(output_path, index=False)

    print(f"Saved LSA candidates to: {output_path}")

    return output_df


# ============================================================
# DOCUMENT-LEVEL MEAN SCORE
# ============================================================

def get_top_source_documents_by_mean_score(
    candidates_df: pd.DataFrame,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    top_n: int = 20,
    min_match_count: int = 4,
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Rank unique source documents by mean LSA score.

    If the same source document appears multiple times because of different
    source chunks, this computes the mean score for that source document.
    """

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "LSA_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates_df: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            mean_LSA_score=("LSA_score", "mean"),
            max_LSA_score=("LSA_score", "max"),
            min_LSA_score=("LSA_score", "min"),
            match_count=("LSA_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = (
        grouped_df
        .sort_values(["mean_LSA_score", "match_count"], ascending=[False, False])
        .head(top_n)
        .reset_index(drop=True)
    )

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "mean_LSA_score",
            "max_LSA_score",
            "min_LSA_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents to: {output_path}")

    return grouped_df


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    BUILD_INDEX=True
    if BUILD_INDEX:
        build_lsa_index(
            source_chunks_path=SOURCE_CHUNKS_PATH,
            artifact_dir=ARTIFACT_DIR,
            n_components=20,
            max_features=20000,
            max_source_chunks=300000,
        )

    candidates_df = retrieve_lsa_candidates_for_suspicious_doc(
        suspicious_chunks_path=SUSPICIOUS_CHUNKS_PATH,
        artifact_dir=ARTIFACT_DIR,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        output_path=OUTPUT_PATH,
        top_k=20,
        batch_size=8,
        max_suspicious_chunks=None,
    )

    top_sources_df = get_top_source_documents_by_mean_score(
        candidates_df=candidates_df,
        source_chunks_path=PROCESSED_DIR / "source_chunks.parquet",
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        top_n=20,
        min_match_count=4,
        output_path=PROCESSED_DIR / "lsa_top_source_documents_by_mean_score.parquet",
    )


print("\nTOP SOURCE DOCUMENTS BY MEAN LSA SCORE")
top_sources_df

Loading LSA artifacts...
Selected suspicious document: part1__suspicious-document00001.txt
Suspicious chunks to query: 22
Searching top-20 source chunks per suspicious chunk
Processed suspicious chunks 0 to 8
Processed suspicious chunks 8 to 16
Processed suspicious chunks 16 to 22
Saved LSA candidates to: ..\datasets\processed\PAN2011_300\lsa_candidates_suspicious_doc_00001.parquet
Saved top source documents to: ..\datasets\processed\PAN2011_300\lsa_top_source_documents_by_mean_score.parquet

TOP SOURCE DOCUMENTS BY MEAN LSA SCORE


,source_doc_rank,source_doc_id,mean_LSA_score,max_LSA_score,min_LSA_score,match_count,unique_source_chunks,unique_suspicious_chunks,source_relative_path
0,1,part10__source-document04575.txt,0.976444,0.983145,0.971190,4,4,2,part10/source-document04575.txt
1,2,part12__source-document05578.txt,0.976094,0.977494,0.973128,4,4,2,part12/source-document05578.txt
2,3,part1__source-document00068.txt,0.974461,0.979856,0.969220,9,9,9,part1/source-document00068.txt
3,4,part1__source-document00009.txt,0.973218,0.980003,0.960249,7,7,3,part1/source-document00009.txt
4,5,part1__source-document00258.txt,0.972225,0.979673,0.966340,4,4,3,part1/source-document00258.txt
5,6,part10__source-document04844.txt,0.971329,0.985869,0.954580,7,7,5,part10/source-document04844.txt
6,7,part1__source-document00255.txt,0.970882,0.985221,0.936579,14,14,6,part1/source-document00255.txt
7,8,part11__source-document05478.txt,0.969085,0.979837,0.945997,18,16,9,part11/source-document05478.txt
8,9,part11__source-document05279.txt,0.968882,0.978354,0.958638,10,9,7,part11/source-document05279.txt
9,10,part12__source-document05627.txt,0.968633,0.976811,0.953938,5,5,5,part12/source-document05627.txt
